<a href="https://colab.research.google.com/github/adamo130-dev/FinRobot/blob/master/FinRobot_Forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio>=3.0 pandas requests openai finnhub-python

In [2]:
%cp /content/drive/MyDrive/OAI_CONFIG_LIST /content/OAI_CONFIG_LIST

In [3]:
%cp /content/drive/MyDrive/config_api_keys /content/config_api_keys

In [4]:
import os, json, time, random
from collections import defaultdict
from datetime import date, datetime, timedelta
import gradio as gr

import pandas as pd
import finnhub
from openai import OpenAI

from io import StringIO
import requests


# ---------- 0  CONFIG ---------------------------------------------------------

def get_openai_api_key(model_name="gpt-4o-mini", config_path="/content/OAI_CONFIG_LIST"):
    with open(config_path, "r") as f:
        configs = json.load(f)
    for entry in configs:
        if entry.get("model") == model_name:
            return entry.get("api_key")
    raise RuntimeError(f"API key for model {model_name} not found in {config_path}")

OPENAI_MODEL  = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_API_KEY = get_openai_api_key(OPENAI_MODEL)
client = OpenAI(api_key=OPENAI_API_KEY)

def load_api_keys(config_path="/content/config_api_keys"):
    try:
        with open(config_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Config file not found at {config_path}. Please ensure it exists.")
        return {}
    except json.JSONDecodeError:
        print(f"Error decoding JSON from {config_path}. Please check the file format.")
        return {}


api_keys = load_api_keys()

FINNHUB_KEY = api_keys.get("finnhub_api_key") or api_keys.get("FINNHUB_API_KEY")

# --- Add your Finnhub API key manually below this line if not found in config_api_keys ---
FINNHUB_KEY = "d0aghfpr01qm3l9l9fagd0aghfpr01qm3l9l9fb0"
# ------------------------------------------------------------------------------------------

if not FINNHUB_KEY:
    print("Finnhub API key not found in config file or environment variables.")
    print("Please manually add your Finnhub API key in the code cell above this message.")
    # You can uncomment the line above and replace "YOUR_FINNHUB_API_KEY" with your actual key.
    # For example: FINNHUB_KEY = "YOUR_FINNHUB_API_KEY"


ALPHA_KEY = os.getenv("ALPHA_KEY")  # If you want to support env override
if not ALPHA_KEY:
    ALPHA_KEY = api_keys.get("alpha_key") or api_keys.get("ALPHA_KEY") # fallback or load from config if you add it there
    if not ALPHA_KEY:
      ALPHA_KEY = "QKAWCJ6Q19ZYBB3S" # Hardcoded fallback

ALPHA_KEY2 = os.getenv("ALPHA_KEY2")
if not ALPHA_KEY2:
    ALPHA_KEY2 = api_keys.get("alpha_key2") or api_keys.get("ALPHA_KEY2")


finnhub_client = finnhub.Client(api_key=FINNHUB_KEY)


SYSTEM_PROMPT = (
    "You are a seasoned stock-market analyst. "
    "Given recent company news and optional basic financials, "
    "return:\n"
    "[Positive Developments] – 2-4 bullets\n"
    "[Potential Concerns] – 2-4 bullets\n"
    "[Prediction & Analysis] – a one-week price outlook with rationale."
)


# ---------- 1  DATE / UTILITY HELPERS ----------------------------------------

def today() -> str:
    return date.today().strftime("%Y-%m-%d")

def n_weeks_before(date_string: str, n: int) -> str:
    return (datetime.strptime(date_string, "%Y-%m-%d") -
            timedelta(days=7 * n)).strftime("%Y-%m-%d")


# ---------- 2  DATA FETCHING --------------------------------------------------

def get_stock_data(symbols: list[str], steps: list[str]) -> pd.DataFrame:

    if not ALPHA_KEY:
        raise RuntimeError("ALPHAVANTAGE_API_KEY is Missing")

    if len(symbols) > 25 and not ALPHA_KEY2:
        raise RuntimeError("ALPHA_KEY2 is required for more than 25 stocks")

    all_data = pd.DataFrame()

    for i, symbol in enumerate(symbols):
        # Determine which API key to use
        current_alpha_key = ALPHA_KEY
        if len(symbols) > 25 and i >= 25:
            current_alpha_key = ALPHA_KEY2
            if not current_alpha_key:
                 raise RuntimeError(f"ALPHA_KEY2 not available for symbol {symbol}")


        # 免费端点：TIME_SERIES_DAILY
        url = (
            "https://www.alphavantage.co/query"
            "?function=TIME_SERIES_DAILY"
            f"&symbol={symbol}"
            f"&apikey={current_alpha_key}"
            "&datatype=csv"
            "&outputsize=full"
        )

        # 重试 3 次
        text = None
        for attempt in range(3):
            resp = requests.get(url, timeout=10)
            if not resp.ok:
                time.sleep(1)
                continue
            text = resp.text.strip()
            if text.startswith("{"):
                info = resp.json()
                msg = info.get("Note") or info.get("Error Message") or str(info)
                # Check if the error message indicates an invalid API key
                if "Invalid API key" in msg:
                    raise RuntimeError(f"Alpha Vantage API Key Error for {symbol} with key {current_alpha_key}: {msg}")
                else:
                    raise RuntimeError(f"Alpha Vantage Return Error for {symbol}: {msg}")
            break

        if not text:
            raise RuntimeError(f"Alpha Vantage Connection Error for {symbol}: {url}")

        df = pd.read_csv(StringIO(text))
        date_col = "timestamp" if "timestamp" in df.columns else df.columns[0]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.sort_values(date_col).set_index(date_col)

        data = {"Symbol": [], "Start Date": [], "End Date": [], "Start Price": [], "End Price": []}
        for j in range(len(steps) - 1):
            s_date = pd.to_datetime(steps[j])
            e_date = pd.to_datetime(steps[j+1])
            seg = df.loc[s_date:e_date]
            if seg.empty:
                 # Check if the segment is empty due to data not being available yet for the dates
                latest_date = df.index.max()
                if latest_date < s_date:
                     print(f"Warning: No data available for {symbol} on or after {s_date}. Latest data is from {latest_date}. Skipping analysis for this symbol.")
                     continue # Skip this symbol if no data for the period
                else:
                    raise RuntimeError(
                        f"Alpha Vantage 无法获取 {symbol} 在 {steps[j]} – {steps[j+1]} 的数据"
                    )
            data["Symbol"].append(symbol)
            data["Start Date"].append(seg.index[0])
            data["Start Price"].append(seg["close"].iloc[0])
            data["End Date"].append(seg.index[-1])
            data["End Price"].append(seg["close"].iloc[-1])

        if data["Symbol"]: # Only append if data was found for at least one segment
          all_data = pd.concat([all_data, pd.DataFrame(data)], ignore_index=True)

        # Limits：5 times/min per key
        time.sleep(12)

    return all_data


def current_basics(symbol: str, curday: str) -> dict:
    raw = finnhub_client.company_basic_financials(symbol, "all")
    if not raw["series"]:
        return {}
    merged = defaultdict(dict)
    for metric, vals in raw["series"]["quarterly"].items():
        for v in vals:
            merged[v["period"]][metric] = v["v"]

    latest = max((p for p in merged if p <= curday), default=None)
    if latest is None:
        return {}
    d = dict(merged[latest])
    d["period"] = latest
    return d

def attach_news(df: pd.DataFrame) -> pd.DataFrame:
    news_col = []
    for _, row in df.iterrows():
        symbol = row["Symbol"]
        start = row["Start Date"].strftime("%Y-%m-%d")
        end   = row["End Date"].strftime("%Y-%m-%d")
        time.sleep(1)                                        # Finnhub QPM guard
        weekly = finnhub_client.company_news(symbol, _from=start, to=end)
        weekly_fmt = [
            {
                "date"    : datetime.fromtimestamp(n["datetime"]).strftime("%Y%m%d%H%M%S"),
                "headline": n["headline"],
                "summary" : n["summary"],
            }
            for n in weekly
        ]
        weekly_fmt.sort(key=lambda x: x["date"])
        news_col.append(json.dumps(weekly_fmt))
    df["News"] = news_col
    return df


# ---------- 3  PROMPT CONSTRUCTION -------------------------------------------

def sample_news(news: list[str], k: int = 5) -> list[str]:
    if len(news) <= k: return news
    return [news[i] for i in sorted(random.sample(range(len(news)), k))]


def make_prompt(symbol: str, df_symbol: pd.DataFrame, curday: str, use_basics=False) -> str:
    # Company profile
    prof = finnhub_client.company_profile2(symbol=symbol)
    company_blurb = (
        f"[Company Introduction]:\n{prof.get('name', 'N/A')} operates in the "
        f"{prof.get('finnhubIndustry', 'N/A')} sector ({prof.get('country', 'N/A')}). "
        f"Founded {prof.get('ipo', 'N/A')}, market cap {prof.get('marketCapitalization', 0.0):.1f} "
        f"{prof.get('currency', 'N/A')}; ticker {symbol} on {prof.get('exchange', 'N/A')}.\n"
    )

    # Past weeks block
    past_block = ""
    for _, row in df_symbol.iterrows():
        term = "increased" if row["End Price"] > row["Start Price"] else "decreased"
        head = (f"From {row['Start Date']:%Y-%m-%d} to {row['End Date']:%Y-%m-%d}, "
                f"{symbol}'s stock price {term} from "
                f"{row['Start Price']:.2f} to {row['End Price']:.2f}.")
        news_items = json.loads(row["News"])
        summaries  = [
            f"[Headline] {n['headline']}\n[Summary] {n['summary']}\n"
            for n in news_items
            if not n["summary"].startswith("Looking for stock market analysis")
        ]
        past_block += "\n" + head + "\n" + "".join(sample_news(summaries, 5))

    # Optional basic financials
    if use_basics:
        basics = current_basics(symbol, curday)
        if basics:
            basics_txt = "\n".join(f"{k}: {v}" for k, v in basics.items() if k != "period")
            basics_block = (f"\n[Basic Financials] (reported {basics['period']}):\n{basics_txt}\n")
        else:
            basics_block = "\n[Basic Financials]: not available\n"
    else:
        basics_block = "\n[Basic Financials]: not requested\n"

    horizon = f"{curday} to {n_weeks_before(curday, -1)}"
    final_user_msg = (
        company_blurb
        + past_block
        + basics_block
        + f"\nBased on all information before {curday}, analyse positive "
          "developments and potential concerns for {symbol}, then predict its "
          f"price movement for next week ({horizon})."
    )
    return final_user_msg


# ---------- 4  LLM CALL -------------------------------------------------------

def chat_completion(prompt: str,
                    model: str = OPENAI_MODEL,
                    temperature: float = 0.3,
                    stream: bool = False) -> str:

    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        stream=stream,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt}
        ],
    )

    if stream:
        collected = []
        for chunk in response:
            delta = chunk.choices[0].delta.content or ""
            print(delta, end="", flush=True)
            collected.append(delta)
        print()
        return "".join(collected)

    # without stream
    return response.choices[0].message.content


# ---------- 5  MAIN ENTRY (CLI test) -----------------------------------------

def predict_single_symbol(symbol: str,
                          curday: str,
                          n_weeks: int,
                          use_basics: bool) -> tuple[str, str]:
    steps = [n_weeks_before(curday, n) for n in range(n_weeks + 1)][::-1]
    # get_stock_data now takes a list of symbols and returns a DataFrame with a 'Symbol' column
    df = get_stock_data([symbol], steps)
    df = attach_news(df)

    if df.empty:
        return f"No data or news available for {symbol}.", ""

    # Filter the DataFrame for the current symbol
    df_symbol = df[df["Symbol"] == symbol].copy()

    prompt_info = make_prompt(symbol, df_symbol, curday, use_basics)
    answer      = chat_completion(prompt_info, stream=False)

    return prompt_info, answer


# ---------- 6  SETUP HF -----------------------------------------


def hf_predict(symbols_string, n_weeks, use_basics):
    symbols = [s.strip().upper() for s in symbols_string.split(',') if s.strip()]
    if len(symbols) > 50:
        return "Error: Maximum of 50 tickers allowed.", "", gr.update(visible=False), gr.update(visible=False)

    results = []
    curday = date.today().strftime("%Y-%m-%d")

    for symbol in symbols:
        try:
            prompt, answer = predict_single_symbol(
                symbol=symbol,
                curday=curday,
                n_weeks=int(n_weeks),
                use_basics=bool(use_basics)
            )
            results.append(f"--- Analysis for {symbol} ---\n\nPrompt:\n{prompt}\n\nAnswer:\n{answer}\n")
        except Exception as e:
            results.append(f"--- Error analyzing {symbol} ---\n{e}\n")

    analysis_text = "\n".join(results)
    # After successful analysis, make the buttons visible and return analysis text
    return analysis_text, gr.update(visible=True), gr.update(visible=True)

# The google.colab package is not available in the standard Python environment
# but is available in the Colab environment where this code is expected to run.
# To avoid errors in environments without google.colab, we can conditionally import it
# or handle the potential NameError if it's used directly.
try:
    from google.colab import drive
    drive_mounted = False
except ImportError:
    drive = None
    drive_mounted = False
    print("google.colab not found. 'Save to Google Drive' functionality will be unavailable.")


def save_to_drive(analysis_text):
    """Saves the provided analysis text to a file in Google Drive."""
    global drive_mounted

    if drive is None:
        return "Error: google.colab not available. Cannot save to Google Drive."

    try:
        # Mount Google Drive if not already mounted
        if not drive_mounted:
            drive.mount('/content/drive')
            drive_mounted = True
            print("Google Drive mounted.")

        # Define the directory and filename
        drive_dir = '/content/drive/My Drive/FinRobot_AnalysisReports'
        # Create the directory if it doesn't exist
        os.makedirs(drive_dir, exist_ok=True)

        # Create a unique filename using timestamp
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"analysis_{timestamp}.txt"
        file_path = os.path.join(drive_dir, filename)

        # Write the analysis text to the file
        with open(file_path, 'w') as f:
            f.write(analysis_text)

        return f"Analysis saved to Google Drive at: {file_path}"

    except Exception as e:
        return f"Error saving to Google Drive: {e}"

def download_analysis(analysis_text):
    """Prepares the analysis text for download."""
    # Create a temporary file to store the analysis text
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"analysis_{timestamp}.txt"
    file_path = os.path.join("/tmp", filename) # Use /tmp for temporary files

    with open(file_path, "w") as f:
        f.write(analysis_text)

    # Return the file path. Gradio's File component can handle this.
    return file_path


with gr.Blocks() as demo:
    gr.Markdown("FinRobot_Forecaster")
    with gr.Row():
        symbol = gr.Textbox(label="Tickers (comma-separated, max 50)", value="AAPL,MSFT")
        n_weeks = gr.Slider(1, 6, value=3, step=1, label="Trace Back Weeks")
        use_basics = gr.Checkbox(label="Add Basic Financials", value=False)
    output_prompt = gr.Textbox(label="Model Output", lines=20) # Increased lines for multiple outputs
    # output_answer is no longer needed as a separate output from hf_predict
    status_output = gr.Textbox(label="Status", interactive=False) # Add a status box

    btn = gr.Button("Run Forecaster")

    # Add the new buttons
    save_button = gr.Button("Save to Google Drive", visible=False)
    download_button = gr.Button("Download Analysis", visible=False)

    # Add a File component for download
    download_file = gr.File(label="Download Analysis File", visible=False)


    btn.click(fn=hf_predict,
              inputs=[symbol, n_weeks, use_basics],
              outputs=[output_prompt, save_button, download_button]) # Updated outputs

    # Link the save button to the save_to_drive function
    save_button.click(
        fn=save_to_drive,
        inputs=[output_prompt], # Pass the analysis text from the output textbox
        outputs=[status_output] # Display the save status in the status box
    )

    # Link the download button to the download_analysis function
    download_button.click(
        fn=download_analysis,
        inputs=[output_prompt], # Pass the analysis text from the output textbox
        outputs=[download_file] # Output to the download file component
    )


if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e88b7b43bf60c56c8a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
